# Simple OpenAI Fine-Tune (explained deeply)

Lives in **`15_MODEL_FINE_TUNING/`**. Run locally (this folder), not Colab.

**How this notebook is organized:** **code cell first**, then **explanation** under it (same pattern as tokenizers / quantization / Colab notebooks).

Related:
- [`1_fine_tuning_in_llm.md`](./1_fine_tuning_in_llm.md) — RAG vs fine-tuning (behavior vs knowledge)
- Sample output files: [`jsonl_simple/`](./jsonl_simple/)

Flow you will actually run:

```text
products (Item)
    ↓
chat messages (user question + assistant price)
    ↓
JSONL files (one example per line)
    ↓
(optional, costs money) upload → fine-tune job → wait → ask the new model
```

`DRY_RUN = True` stops **before** any paid OpenAI API calls. JSONL writing is free.

Run **top → bottom**.

## Big picture — what fine-tuning is doing here

The **base model** already speaks English. Fine-tuning does **not** teach it products from scratch.

It shows many examples of:

```text
User:      Estimate the price of this product. ... <product text>
Assistant: $29.99
```

so a **private copy** of the model starts answering in that **style**: a short `$xx.xx` price, not a paragraph.

| Piece | Job |
|-------|-----|
| **Base model** (`OPENAI_MODEL`) | Starting point (already trained) |
| **Train JSONL** | Lessons: question + correct price |
| **Val JSONL** | Held-out checks while OpenAI trains |
| **Fine-tuned model** (`ft:...`) | Private copy that behaves like the lessons |

This matches the notes: **fine-tuning changes behavior** (format / task). RAG would be for looking up changing product catalogs.

OpenAI is winding down self-serve fine-tuning for many orgs (`403 training_not_available`). Steps 1–5 still teach the **data format**. If `jobs.create` is blocked on your account, keep the JSONL lesson and train locally later (LoRA) instead of OpenAI SFT.

## 0) Imports

In [ ]:
import json
import os
import time
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv

### Explanation — imports

#### What this cell does

Load Python tools. Nothing talks to OpenAI yet. No files are written yet.

| Line | Meaning |
|------|---------|
| `import json` | Turn Python dicts into JSON **text** and back. Each JSONL line is `json.dumps(...)`. |
| `import os` | Read environment variables (`os.getenv("OPENAI_MODEL")`, later `OPENAI_API_KEY`). |
| `import time` | `time.sleep(20)` in the wait loop so we do not hammer the API every millisecond. |
| `from dataclasses import dataclass` | Decorator that writes `__init__` for a tiny `Item` class. |
| `from pathlib import Path` | File paths as objects (`JSONL_DIR / "fine_tune_train.jsonl"`) instead of raw strings. |
| `from dotenv import load_dotenv` | Read a local `.env` file into the process environment (keys stay off the notebook). |

`openai` is **not** imported here on purpose. Upload / train cells import it only when `DRY_RUN` is `False`, so a dry run does not even need the client if you only care about JSONL.

If `dotenv` is missing: `pip install python-dotenv` in the same environment you use to run this notebook.

## 1) Config

In [ ]:
load_dotenv(override=True)  # load OPENAI_MODEL / OPENAI_API_KEY from .env

DRY_RUN = True  # flip to False only when you want to spend OpenAI credits

BASE_MODEL = os.getenv("OPENAI_MODEL")
if not BASE_MODEL:
    raise SystemExit("Set OPENAI_MODEL in your .env (e.g. OPENAI_MODEL=gpt-4.1-nano)")

JSONL_DIR = Path.cwd() / "jsonl_simple"

print("DRY_RUN =", DRY_RUN)
print("BASE_MODEL =", BASE_MODEL)
print("JSONL_DIR =", JSONL_DIR)

### Explanation — config

#### What this cell does

```text
.env file  →  environment variables  →  BASE_MODEL
this folder  →  JSONL_DIR = .../jsonl_simple
DRY_RUN flag  →  later cells skip paid API calls
```

| Line | Meaning |
|------|---------|
| `load_dotenv(override=True)` | Read `.env` from the current working directory (or parents). `override=True` means values in `.env` **win** over whatever was already in the shell. |
| `DRY_RUN = True` | Safety switch. `True` = build JSONL only (free). `False` = upload + train + chat (costs credits, needs a fine-tune-enabled org). |
| `BASE_MODEL = os.getenv("OPENAI_MODEL")` | Read the model **name string** from env. Returns `None` if the variable is missing. **No hardcoded model** in the notebook. |
| `if not BASE_MODEL:` | Empty string and `None` both count as missing. |
| `raise SystemExit("...")` | Stop the notebook with a clear message instead of failing much later inside OpenAI. |
| `Path.cwd()` | Current working directory. A notebook has no `__file__`, so we cannot say “folder of this .ipynb” the same way a `.py` script can. Run the kernel **from this folder**. |
| `Path.cwd() / "jsonl_simple"` | Join cwd with the folder name `jsonl_simple`. Result is a `Path`, not yet created on disk. |
| `print(...)` | Confirm the three settings before anything else runs. |

#### `.env` example (same folder or a parent that `load_dotenv` finds)

```env
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4.1-nano
```

| Variable | Used for |
|----------|----------|
| `OPENAI_API_KEY` | Authentication (upload / train / chat). Not printed. |
| `OPENAI_MODEL` | Copied into `BASE_MODEL` and passed to `jobs.create` **exactly as written**. |

Do not paste keys into cells.

## 2) `Item` class

In [ ]:
@dataclass
class Item:
    title: str
    summary: str
    price: float

### Explanation — `Item`

#### What this cell does

Define one product as three fields. This is a tiny stand-in for a real loader like `pricer.items.Item` (Hugging Face catalog, etc.).

| Line | Meaning |
|------|---------|
| `@dataclass` | Python writes `__init__`, `__repr__`, etc. for you. After this, `Item("Kettle", "1.7L...", 29.99)` works. |
| `class Item:` | Name of the type. Each product in `TRAIN` / `VAL` is one `Item`. |
| `title: str` | Human label for prints (`"Stainless Steel Kettle"`). The model is **not** trained on title in our message helper — only `summary` goes into the user text. |
| `summary: str` | Product description the model **reads** as the question. |
| `price: float` | Correct answer the assistant should learn (`29.99` → `"$29.99"`). |

Without `@dataclass` you would write a boring `__init__` that copies those three arguments onto `self`. Same idea, more noise.

## 3) Fake train / val data

In [ ]:
TRAIN = [
    Item("Stainless Steel Kettle", "1.7L electric kettle, boil-dry protection, black", 29.99),
    Item("Wireless Mouse", "Ergonomic USB-C rechargeable mouse, silent clicks", 18.50),
    Item("Yoga Mat", "6mm non-slip TPE yoga mat with carrying strap", 24.00),
    Item("USB-C Hub", "7-in-1 hub: HDMI, SD, USB 3.0, 100W PD", 39.99),
    Item("Desk Lamp", "LED desk lamp, 5 brightness levels, USB port", 22.00),
]

VAL = [
    Item("Bluetooth Speaker", "Portable waterproof speaker, 12h battery", 34.99),
    Item("Phone Stand", "Adjustable aluminum stand for phones and tablets", 15.00),
]

print(f"train={len(TRAIN)}, val={len(VAL)}")
print("First train item:", TRAIN[0])


### Explanation — datasets

#### What this cell does

```text
TRAIN  →  5 products the trainer may update weights from
VAL    →  2 products used to check (not the same titles)
```

A real pricing project would load thousands of rows (`Item.from_hub(...)`). Here the list is hardcoded so you can **see** every example.

| Line | Meaning |
|------|---------|
| `TRAIN = [` | Start a Python list of training `Item`s. |
| `Item("Stainless Steel Kettle", "1.7L ...", 29.99)` | One product: title, summary, price. Same shape for every row. |
| `VAL = [` | Second list. **Different products** so validation is not a copy of train. |
| `len(TRAIN)` / `len(VAL)` | Count examples (expect `5` and `2`). |
| `TRAIN[0]` | First item. `@dataclass` `__repr__` prints fields so you can sanity-check. |
| `print(f"train={len(TRAIN)}, ...")` | f-string: insert the counts into the message. |

OpenAI often suggests on the order of **tens to hundreds** of examples for a real job. **5 + 2 is only for learning the pipeline.** A model trained on this tiny set will not become a good pricer; it will show the **file format** and API steps.

Train vs val: weights update from **train**. Val is a report card during training. Do not put the same product in both.

## 4) Chat message helpers

In [ ]:
def messages_for(item: Item) -> list[dict]:
    """Training example: user question + correct assistant answer."""
    user = (
        "Estimate the price of this product. "
        f"Respond with the price, no explanation\n\n{item.summary}"
    )
    return [
        {"role": "user", "content": user},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]


def test_messages_for(item: Item) -> list[dict]:
    """Inference: only the user message (no answer)."""
    user = (
        "Estimate the price of this product. "
        f"Respond with the price, no explanation\n\n{item.summary}"
    )
    return [{"role": "user", "content": user}]


print(json.dumps(messages_for(TRAIN[0]), indent=2))

### Explanation — messages

#### What this cell does

Fine-tuning chat models = show **conversations**, not a CSV of `summary,price`.

```text
messages_for(item)       →  [user, assistant]   ← lesson (JSONL)
test_messages_for(item)  →  [user]              ← exam (section 9)
```

| Line | Meaning |
|------|---------|
| `def messages_for(item: Item) -> list[dict]:` | Function: take one `Item`, return a list of message dicts. `-> list[dict]` is a type hint (documentation for humans/tools). |
| `"""Training example: ..."""` | Docstring. Explains **when** to use this function. |
| `user = (` ... `)` | Build the **user** string. Parentheses join adjacent string literals. |
| `"Estimate the price..."` | Fixed instruction: what task + “no explanation”. |
| `f"...\n\n{item.summary}"` | f-string: insert this product’s `summary`. `\n\n` is a blank line between instruction and product text. |
| `return [` | Start the messages list OpenAI expects. |
| `{"role": "user", "content": user}` | First turn: the question. `role` must be one of the chat roles (`user` / `assistant` / `system`). |
| `{"role": "assistant", "content": f"${item.price:.2f}"}` | Second turn: the **correct** answer. `:.2f` means 2 decimal places (`29.9` → `29.90`). `$` is a literal dollar sign in the string. |
| `def test_messages_for(...)` | Same user text, **no** assistant message. At inference you must not leak the true price. |
| `return [{"role": "user", "content": user}]` | One-element list. |
| `json.dumps(..., indent=2)` | Pretty-print the first train example as JSON so you can **see** the structure in the output. `indent=2` is only for display — JSONL files later stay **one compact line**. |
| `messages_for(TRAIN[0])` | Run the helper on the kettle. |

The user wording is **identical** in train and test. Only the assistant line is added for training. If you change the prompt in one function and not the other, the fine-tuned model sees a different exam than the lessons.

## 5) Write JSONL files

In [ ]:
def write_jsonl(items: list[Item], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for item in items:
            row = {"messages": messages_for(item)}
            f.write(json.dumps(row) + "\n")
    print(f"Wrote {len(items)} examples -> {path}")


def show_first_line(path: Path) -> None:
    first = path.read_text(encoding="utf-8").splitlines()[0]
    print("\nExample JSONL line:")
    print(json.dumps(json.loads(first), indent=2))


train_path = JSONL_DIR / "fine_tune_train.jsonl"
val_path = JSONL_DIR / "fine_tune_validation.jsonl"

write_jsonl(TRAIN, train_path)
write_jsonl(VAL, val_path)
show_first_line(train_path)

### Explanation — write JSONL files

Python `Item` objects live in memory. OpenAI (and most trainers) need a **text file** of examples. That file format is **JSONL**, not a normal JSON array.

#### JSON vs JSONL (the only difference that matters)

**JSON** = one big structure. Usually a list in `[ ]`, with commas between items. You must load the **whole file** to read one example.

```json
[
  {"messages": [ ... kettle example ... ]},
  {"messages": [ ... mouse example ... ]}
]
```

**JSONL** = JSON **Lines**. Each line is a **complete JSON object**. No outer `[ ]`. No comma after each line. Press Enter after every example.

```text
{"messages": [ ... kettle example ... ]}
{"messages": [ ... mouse example ... ]}
```

| | JSON | JSONL |
|---|---|---|
| Wrapper | `[ ... ]` + commas | none |
| One example | one element in the list | **one line** |
| Easy to append / stream | no | yes |
| Fine-tune upload | wrong format | **this is what we write** |

The **L** in JSONL is **Lines**.

#### What one training line must look like

Chat fine-tuning wants a dict with one key: `messages`.
That value is the same list `messages_for(item)` already built (user question + assistant price).

So one line is:

```json
{"messages": [{"role": "user", "content": "Estimate the price...\n\n1.7L electric kettle..."}, {"role": "assistant", "content": "$29.99"}]}
```

Five train items → **five lines** in `fine_tune_train.jsonl`.  
Two val items → **two lines** in `fine_tune_validation.jsonl`.

Train file = examples used to teach.  
Val file = examples used to check while training. Same format, different products.

#### Walk the code, every line

`write_jsonl` does **not** hardcode `jsonl_simple`. Config set `JSONL_DIR`; this cell only **uses** it.

| Line | Meaning |
|------|---------|
| `def write_jsonl(items, path) -> None:` | `items` = `TRAIN` or `VAL`. `path` = full file path. `None` = writes to disk, returns nothing useful. |
| `path.parent` | Folder that **contains** the file. For `.../jsonl_simple/fine_tune_train.jsonl` this is `.../jsonl_simple`. |
| `path.parent.mkdir(parents=True, exist_ok=True)` | Create that folder if missing. `parents=True` also creates missing parents. `exist_ok=True` = if the folder already exists, do not error. |
| `with path.open("w", encoding="utf-8") as f:` | Open the **file** for write. `"w"` **overwrites** an old file. `utf-8` is the text encoding. `with` closes the file even if a later line fails. |
| `for item in items:` | One loop pass per product. |
| `row = {"messages": messages_for(item)}` | Wrap the chat list in the required key. |
| `json.dumps(row)` | Dict → **one line** of JSON text. **No** `indent=` — extra newlines would break “one object per line”. |
| `f.write(... + "\n")` | Write that text, then a newline so the **next** example is the **next** line. |
| `print(f"Wrote {len(items)} examples -> {path}")` | Confirm count and destination. |
| `def show_first_line(path):` | Read-only helper for humans. |
| `path.read_text(encoding="utf-8")` | Load the whole file as one string. |
| `.splitlines()[0]` | Split on newlines; take the first line (first example). |
| `json.loads(first)` | Parse that line back into a Python dict (proves it is valid JSON). |
| `json.dumps(..., indent=2)` | Pretty-print for the notebook output only. |
| `train_path = JSONL_DIR / "fine_tune_train.jsonl"` | Join folder + filename. |
| `val_path = JSONL_DIR / "fine_tune_validation.jsonl"` | Same for validation. |
| `write_jsonl(TRAIN, train_path)` | Write 5 lines. |
| `write_jsonl(VAL, val_path)` | Write 2 lines. |
| `show_first_line(train_path)` | Show the kettle line pretty-printed. |

Open the files in the editor: one compact JSON object per line, one product per line.

## Checkpoint

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True -> stopping before OpenAI calls.")
    print("Open the files in jsonl_simple/, then set DRY_RUN=False to continue.")
else:
    print("DRY_RUN=False -> continue with the OpenAI cells below.")

### Explanation — checkpoint

#### What this cell does

Print which road you are on. It does **not** `raise` or stop the kernel. Later cells each wrap themselves in `if not DRY_RUN:` so they skip paid work.

| Line | Meaning |
|------|---------|
| `if DRY_RUN:` | Same flag as Config. |
| `print("DRY_RUN=True -> ...")` | Reminder: JSONL is done; do not expect upload/train to run. |
| `print("Open the files...")` | Next learning step: actually look at `jsonl_simple/`. |
| `else:` | You flipped the flag and re-ran from Config downward. |
| `print("DRY_RUN=False -> ...")` | Upload / job / wait / test cells will execute. |

You already learned: **products → chat messages → JSONL**.

Everything below is **upload + train + ask** (money + an org that still allows fine-tuning).

## 6) Upload JSONL to OpenAI

> Skip while `DRY_RUN=True`.

In [ ]:
if not DRY_RUN:
    from openai import OpenAI

    load_dotenv(override=True)
    if not os.getenv("OPENAI_API_KEY"):
        raise SystemExit("Missing OPENAI_API_KEY in environment / .env")

    client = OpenAI()

    with train_path.open("rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")
    with val_path.open("rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")

    print("train file id:", train_file.id)
    print("val   file id:", val_file.id)
else:
    print("Skipped upload (DRY_RUN=True)")

### Explanation — upload

#### What this cell does

```text
local jsonl_simple/*.jsonl  →  OpenAI Files API  →  file ids (file-...)
```

Training does **not** read your laptop path. OpenAI only sees files you **upload**.

| Line | Meaning |
|------|---------|
| `if not DRY_RUN:` | Run this block only when you opted into paid API. |
| `from openai import OpenAI` | Official client. Imported here so dry-run notebooks do not require it to understand JSONL. |
| `load_dotenv(override=True)` | Reload `.env` in case you added the key after the Config cell. |
| `if not os.getenv("OPENAI_API_KEY"):` | Key missing → fail now, not with a cryptic 401 later. |
| `raise SystemExit(...)` | Stop with a readable message. |
| `client = OpenAI()` | Create a client. **No** `api_key=` argument: it reads `OPENAI_API_KEY` from the environment. Default host is OpenAI cloud. |
| `train_path.open("rb")` | Open the train JSONL as **bytes**. Upload APIs want a binary file handle, not a decoded string. `"rb"` = read binary. |
| `with ... as f:` | Close the handle after upload. |
| `client.files.create(file=f, purpose="fine-tune")` | POST the bytes. `purpose="fine-tune"` tells OpenAI this file is for training jobs (not assistants, not batch). |
| `train_file = ...` | Response object. `.id` is a string like `file-abc123`. |
| same for `val_path` / `val_file` | Second upload. |
| `print(..., train_file.id)` | You need these ids in the next cell. |
| `else: print("Skipped upload...")` | Dry-run path: no network. |

`train_file` / `val_file` only exist if this block ran. Section 7 uses `.id`, not `train_path`.

## 7) Start fine-tune job

In [ ]:
if not DRY_RUN:
    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        suffix="pricer-demo",
    )
    print("job id:", job.id)
    print("status:", job.status)
else:
    print("Skipped fine-tune create (DRY_RUN=True)")

### Explanation — start the fine-tune job

#### What this cell does

Section 6 **stored** files. This cell **asks OpenAI to train** a private copy of `BASE_MODEL` on those files.

```text
jobs.create(...)  →  job id (ftjob-...)
                    training happens later on OpenAI GPUs
```

This is **not** training on your laptop. The call returns when the job is **created**, often still `validating_files` / `queued` — not `succeeded`.

| Line | Meaning |
|------|---------|
| `if not DRY_RUN:` | Same money/API gate. |
| `client.fine_tuning.jobs.create(` | HTTP request: start a fine-tune job. Uses `client` from section 6 (same kernel). |
| `training_file=train_file.id` | OpenAI **file id** from upload, **not** `jsonl_simple/fine_tune_train.jsonl`. Teaching examples. |
| `validation_file=val_file.id` | Held-out file id. OpenAI scores on these. Weights update from **train**, not val. |
| `model=BASE_MODEL` | Starting checkpoint from `.env` `OPENAI_MODEL`. You customize this model; you do not train from random weights. |
| `seed=42` | Fixes shuffle / some randomness so a re-run is more repeatable. |
| `hyperparameters={...}` | Extra knobs. Dict with two keys below. |
| `"n_epochs": 1` | How many times the trainer walks through the **entire** train file. `1` = see each example once (cheap, less overfitting on 5 rows). |
| `"batch_size": 1` | How many examples before one weight update. `1` = update after every example (fine for a handful of rows). |
| `suffix="pricer-demo"` | Label baked into the finished model name, e.g. `ft:...:pricer-demo:...`, so you can find it in the dashboard. |
| `job = ...` | Job object. |
| `print("job id:", job.id)` | e.g. `ftjob-...`. Section 8 polls this id. |
| `print("status:", job.status)` | Snapshot **now** — usually not `succeeded` yet. |
| `else: print("Skipped...")` | Dry run. |

| Status (typical) | Meaning |
|------------------|--------|
| `validating_files` | checking JSONL format |
| `queued` / `running` | actually training |
| `succeeded` | done — then `job.fine_tuned_model` is set |
| `failed` / `cancelled` | no usable model |

**Start ≠ finish.** This cell only **creates** the job.

#### If you get `403 training_not_available`

OpenAI is winding down self-serve fine-tuning for many orgs. That is **not** a bug in this notebook.

- Error: `... no longer able to create new fine-tuning training jobs ... code: training_not_available`
- Docs: https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning

Steps 1–5 (JSONL) still teach the idea. Upload may work; **this `jobs.create` call will fail** on a blocked account. Then use local LoRA, or prompts / RAG, instead of OpenAI SFT.

## 8) Wait for job

In [ ]:
if not DRY_RUN:
    while True:
        job = client.fine_tuning.jobs.retrieve(job.id)
        print("status:", job.status)
        if job.status in {"succeeded", "failed", "cancelled"}:
            break
        time.sleep(20)

    if job.status != "succeeded":
        raise SystemExit(f"Fine-tune ended with status={job.status}")

    model_name = job.fine_tuned_model
    print("Fine-tuned model:", model_name)
else:
    print("Skipped wait loop (DRY_RUN=True)")

### Explanation — wait loop

#### What this cell does

```text
loop:  retrieve(job.id)  →  print status
         succeeded / failed / cancelled  →  stop looping
         otherwise  →  sleep 20s  →  retrieve again
```

You could re-run `retrieve` by hand. The loop does that for you.

| Line | Meaning |
|------|---------|
| `if not DRY_RUN:` | Same gate. Needs `client` and `job` from previous cells (same kernel). |
| `while True:` | Infinite loop until `break`. |
| `client.fine_tuning.jobs.retrieve(job.id)` | GET latest job object from OpenAI. Overwrite local `job` with a fresh snapshot. |
| `print("status:", job.status)` | See progress in the notebook. |
| `if job.status in {"succeeded", "failed", "cancelled"}:` | Set of **terminal** statuses. `in { ... }` is a membership test. |
| `break` | Leave the `while` loop. |
| `time.sleep(20)` | Pause 20 seconds so we do not spam retrieve. Training can take minutes. |
| `if job.status != "succeeded":` | After the loop, anything other than success is a dead end for section 9. |
| `raise SystemExit(f"Fine-tune ended with status={job.status}")` | Stop; f-string inserts the actual status (`failed`, `cancelled`, ...). |
| `model_name = job.fine_tuned_model` | New model id, usually starts with `ft:`. Only filled when the job succeeded. |
| `print("Fine-tuned model:", model_name)` | Copy this if you want to call it later from another script. |
| `else: print("Skipped wait loop...")` | Dry run. |

`model_name` is what section 9 passes to `chat.completions.create` — **not** `BASE_MODEL` (that was the starting point).

## 9) Test the fine-tuned model

In [ ]:
if not DRY_RUN:
    sample = VAL[0]
    response = client.chat.completions.create(
        model=model_name,
        messages=test_messages_for(sample),
        max_tokens=7,
    )
    guess = response.choices[0].message.content

    print("Product:", sample.title)
    print(f"True price: ${sample.price:.2f}")
    print("Model guess:", guess)
else:
    print("Skipped prediction (DRY_RUN=True)")

### Explanation — test

#### What this cell does

Normal **chat completion**, but `model=` is the new `ft:...` name.

```text
VAL[0] summary  →  user message only  →  fine-tuned model  →  guess string
compare guess to sample.price  (we print both; we do not score automatically)
```

| Line | Meaning |
|------|---------|
| `if not DRY_RUN:` | Same gate. Needs `client` and `model_name`. |
| `sample = VAL[0]` | First validation product (Bluetooth speaker). Held-out vs train titles. Still a **tiny** set — a good guess here is luck + format learning, not a real pricer. |
| `client.chat.completions.create(` | Same API as everyday ChatGPT-style calls. |
| `model=model_name` | The **fine-tuned** id from section 8 (`ft:...`). Using `BASE_MODEL` here would test the **un**-tuned model. |
| `messages=test_messages_for(sample)` | **User only**. If you passed `messages_for(sample)` you would include the true `$34.99` and the test would be fake. |
| `max_tokens=7` | Cap the reply length. A price like `$34.99` is a few tokens; this stops a rambling essay if the model ignores the instruction. |
| `response = ...` | Full API object (choices, usage, …). |
| `response.choices[0]` | First (only) completion. |
| `.message.content` | The assistant text string (the guess). |
| `guess = ...` | Store that string for printing. |
| `print("Product:", sample.title)` | Human label (not sent in the prompt). |
| `print(f"True price: ${sample.price:.2f}")` | Ground truth, same 2-decimal format as training. |
| `print("Model guess:", guess)` | What the model actually said. Compare by eye. |
| `else: print("Skipped prediction...")` | Dry run. |

---

**One-sentence summary:** Fine-tuning here = turn products into Q&A chats, save them as JSONL (one example per line), optionally upload and train a private copy of a base model, then ask that copy new questions **without** giving the answer.